# DSA 4060 | Week 1 Practical
## Build and Publish a Popularity Based Movie Recommender

**Student:** Halima Mahdi  
**Student Number:** [Enter your student number]  
**Date:** [Enter date]

This notebook uses the MovieLens latest-small dataset to inspect user-item ratings, calculate movie popularity, build a minimum-rating popularity recommender and a weighted-rating recommender, test the results, and discuss the strengths and limitations of a non-personalized baseline.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Define File Paths

In [ ]:
# Works when VS Code starts Jupyter in either the project root or notebooks folder.
if Path("data/movies.csv").exists():
    PROJECT_ROOT = Path(".")
elif Path("../data/movies.csv").exists():
    PROJECT_ROOT = Path("..")
else:
    raise FileNotFoundError(
        "Place movies.csv and ratings.csv inside the project's data folder."
    )

DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = PROJECT_ROOT / "images"
IMAGES_DIR.mkdir(exist_ok=True)

MOVIES_FILE = DATA_DIR / "movies.csv"
RATINGS_FILE = DATA_DIR / "ratings.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("Movies file:", MOVIES_FILE.resolve())
print("Ratings file:", RATINGS_FILE.resolve())

## 3. Load the CSV Files

In [ ]:
movies = pd.read_csv(MOVIES_FILE)
ratings = pd.read_csv(RATINGS_FILE)

print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)

## 4. Preview and Inspect

In [ ]:
display(movies.head())
display(ratings.head())

print("Movies information:")
movies.info()

print("\nRatings information:")
ratings.info()

## 5. Validate the Data

In [ ]:
print("Missing values in movies:")
display(movies.isna().sum())

print("Missing values in ratings:")
display(ratings.isna().sum())

print("Duplicate movie rows:", movies.duplicated().sum())
print("Duplicate rating rows:", ratings.duplicated().sum())

print("Rating range:", ratings["rating"].min(), "to", ratings["rating"].max())
print("Unknown movie IDs:",
      (~ratings["movieId"].isin(movies["movieId"])).sum())

### Interpretation
Review the validation output. Do not remove rows simply because a duplicate count is non-zero; interpret duplicates according to the dataset definition.

## 6. Summarize the Interaction Data

In [ ]:
n_users = ratings["userId"].nunique()
n_movies_rated = ratings["movieId"].nunique()
n_movies_total = movies["movieId"].nunique()
n_interactions = len(ratings)

print("Unique users:", n_users)
print("Movies in movies.csv:", n_movies_total)
print("Movies with at least one rating:", n_movies_rated)
print("Total rating interactions:", n_interactions)

display(ratings["rating"].describe())

## 7. Ratings per User and per Movie

In [ ]:
ratings_per_user = ratings.groupby("userId").size()
ratings_per_movie = ratings.groupby("movieId").size()

print("Ratings per user:")
display(ratings_per_user.describe())

print("Ratings per movie:")
display(ratings_per_movie.describe())

## 8. Rating Distribution

In [ ]:
rating_distribution = ratings["rating"].value_counts().sort_index()
display(rating_distribution)

ax = rating_distribution.plot(kind="bar", figsize=(8, 4))
ax.set_title("Distribution of Movie Ratings")
ax.set_xlabel("Rating")
ax.set_ylabel("Number of Ratings")
plt.tight_layout()
plt.show()

### Interpretation
State which rating occurs most often and whether low ratings are relatively rare or common, based only on the chart.

## 9. Interaction Matrix Sparsity

In [ ]:
possible_interactions = n_users * n_movies_rated
observed_interactions = n_interactions
sparsity = 1 - (observed_interactions / possible_interactions)

print(f"Possible interactions: {possible_interactions:,}")
print(f"Observed interactions: {observed_interactions:,}")
print(f"Sparsity: {sparsity:.2%}")

### Interpretation
The sparsity percentage represents the proportion of possible user-movie pairs for which no rating is observed.

## 10. Popularity Recommender — Aggregate Ratings

In [ ]:
movie_stats = (
    ratings.groupby("movieId")
    .agg(
        average_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

movie_stats = movie_stats.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left",
    validate="one_to_one"
)

display(movie_stats.head())

## 11. Most Rated Movies

In [ ]:
most_rated = (
    movie_stats
    .sort_values(
        ["rating_count", "average_rating"],
        ascending=[False, False]
    )
    .head(10)
)

display(most_rated[["title", "rating_count", "average_rating"]])

## 12. Highest Average Ratings

In [ ]:
highest_average = (
    movie_stats
    .sort_values(
        ["average_rating", "rating_count"],
        ascending=[False, False]
    )
    .head(10)
)

display(highest_average[["title", "rating_count", "average_rating"]])

## 13. Minimum-Rating Popularity Baseline

In [ ]:
MIN_RATINGS = 50

popular_movies = (
    movie_stats[movie_stats["rating_count"] >= MIN_RATINGS]
    .sort_values(
        ["average_rating", "rating_count"],
        ascending=[False, False]
    )
    .head(10)
)

display(
    popular_movies[
        ["title", "genres", "rating_count", "average_rating"]
    ]
)

## 14. Recommendation Function

In [ ]:
def recommend_popular_movies(stats, min_ratings=50, top_n=10):
    candidates = stats[stats["rating_count"] >= min_ratings].copy()
    return (
        candidates
        .sort_values(
            ["average_rating", "rating_count"],
            ascending=[False, False]
        )
        .head(top_n)
        [["movieId", "title", "genres", "rating_count", "average_rating"]]
    )

recommendations = recommend_popular_movies(movie_stats, 50, 10)
display(recommendations)

## 15. Weighted-Rating Baseline

Formula:

**Weighted score = (v / (v + m)) × R + (m / (v + m)) × C**

Where:
- R = movie average rating
- v = number of ratings
- C = overall mean rating
- m = minimum evidence threshold

### 15.1 Select the Threshold

In [ ]:
C = ratings["rating"].mean()
m = movie_stats["rating_count"].quantile(0.90)

print(f"Overall mean rating C: {C:.3f}")
print(f"90th percentile rating count m: {m:.1f}")

qualified_count = (movie_stats["rating_count"] >= m).sum()
print("Movies meeting the 90th-percentile threshold:", qualified_count)

### 15.2 Calculate Weighted Scores

In [ ]:
qualified = movie_stats[movie_stats["rating_count"] >= m].copy()

v = qualified["rating_count"]
R = qualified["average_rating"]

qualified["weighted_score"] = (
    (v / (v + m)) * R +
    (m / (v + m)) * C
)

weighted_recommendations = (
    qualified
    .sort_values(
        ["weighted_score", "rating_count"],
        ascending=[False, False]
    )
    .head(10)
)

display(
    weighted_recommendations[
        ["title", "genres", "rating_count",
         "average_rating", "weighted_score"]
    ]
)

## 16. Compare the Two Lists

In [ ]:
comparison = popular_movies[
    ["title", "rating_count", "average_rating"]
].copy()
comparison["method"] = "Minimum 50 ratings"

weighted_view = weighted_recommendations[
    ["title", "rating_count", "average_rating"]
].copy()
weighted_view["method"] = "Weighted score"

display(pd.concat([comparison, weighted_view], ignore_index=True))

common_titles = sorted(
    set(popular_movies["title"]).intersection(
        set(weighted_recommendations["title"])
    )
)

print("Movies appearing in both lists:")
for title in common_titles:
    print("-", title)
print("Number of common movies:", len(common_titles))

## 17. Top 10 Visualization

In [ ]:
plot_data = weighted_recommendations.sort_values("weighted_score")

ax = plot_data.plot(
    x="title",
    y="weighted_score",
    kind="barh",
    figsize=(10, 6),
    legend=False
)

ax.set_title("Top 10 Movies by Weighted Rating")
ax.set_xlabel("Weighted score")
ax.set_ylabel("Movie")
plt.tight_layout()

chart_path = IMAGES_DIR / "top10_recommendations.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()

print("Chart saved to:", chart_path.resolve())

## 18. Basic Tests

In [ ]:
assert len(recommendations) <= 10
assert recommendations["movieId"].is_unique
assert recommendations["rating_count"].ge(50).all()
assert recommendations["average_rating"].between(0.5, 5.0).all()
assert recommendations["title"].notna().all()

assert len(weighted_recommendations) <= 10
assert weighted_recommendations["weighted_score"].notna().all()

print("All checks passed.")

## 19. Try Different Parameters

In [ ]:
print("Top 5 with minimum 20 ratings:")
display(recommend_popular_movies(movie_stats, min_ratings=20, top_n=5))

print("Top 5 with minimum 100 ratings:")
display(recommend_popular_movies(movie_stats, min_ratings=100, top_n=5))

## 20. Optional Genre Filter

In [ ]:
def recommend_by_genre(stats, genre, min_ratings=30, top_n=10):
    mask = stats["genres"].str.contains(genre, case=False, na=False)
    return recommend_popular_movies(stats[mask], min_ratings, top_n)

display(recommend_by_genre(movie_stats, "Comedy", 30, 10))

## 21. Key Results

In [ ]:
most_rated_movie = most_rated.iloc[0]
minimum_threshold_movie = popular_movies.iloc[0]
weighted_top_movie = weighted_recommendations.iloc[0]

print("Movie with the most ratings:", most_rated_movie["title"])
print("Rating count:", int(most_rated_movie["rating_count"]))
print("Average rating:", round(most_rated_movie["average_rating"], 3))

print("\nTop movie after 50-rating threshold:",
      minimum_threshold_movie["title"])
print("Rating count:", int(minimum_threshold_movie["rating_count"]))
print("Average rating:",
      round(minimum_threshold_movie["average_rating"], 3))

print("\nTop movie using weighted score:",
      weighted_top_movie["title"])
print("Weighted score:",
      round(weighted_top_movie["weighted_score"], 3))

print("\nMovies qualifying under 90th-percentile threshold:",
      qualified_count)

## 22. Findings and Limitations

### Findings
Use the actual notebook output to describe the main dataset observations. Discuss the rating distribution, users, movies, interactions, and sparsity. Compare the minimum-rating and weighted-rating Top 10 lists and identify movies appearing in both.

The minimum-rating popularity baseline provides a simple and transparent recommendation list. It combines average rating with an evidence requirement so that movies with very few ratings do not automatically dominate. The weighted-rating baseline further balances a movie's own average against the overall mean according to its rating count.

### Limitations
The approach is not personalized because every user receives the same ranked list. It can create popularity bias because already-popular movies receive more exposure. New movies with no ratings have a cold-start problem. The approach also does not model changes in preferences over time, diversity, novelty, or the reasons behind individual ratings.

### Possible improvements
Future versions could use genres or textual features for content-based recommendations, user-rating patterns for collaborative filtering, train/test evaluation, diversity and novelty metrics, catalogue coverage, and time-aware modelling.

## 23. Final Checklist

- [ ] Student name and student number completed
- [ ] All cells run from top to bottom without errors
- [ ] Results are explained in Markdown
- [ ] Top 10 chart saved in `images/`
- [ ] README contains actual findings
- [ ] requirements.txt is present
- [ ] .gitignore is present
- [ ] Dataset attribution and access date included
- [ ] No passwords, tokens, or private data committed
- [ ] GitHub repository contains the required files